# Module 6.4 — GitHub for Portfolios
### Reference & live-demo notebook — AJEBO Finance MFB Home Loan Applications

**Publica Academy Data Analysis Programme · Module 6, Week 7 (Data Storytelling and Visualisation)**

Modules 6.1-6.3 produced a cleaned dataset, honest charts, and a written narrative. This notebook
turns that work into an actual, publishable `README.md` — generated here from real data, not
typed by hand — plus a repeatable data-safety check every trainee should run before uploading
anything to a public repository.

**Learning outcome:** create a repository, upload a documented analysis project with a readable
README, and share a portfolio link an employer can open.

**Dataset:** `ajebo_finance_loan_applications_CLEANED.csv` — the 356-row output of the Module 6.1
cleaning pipeline.

## Setup

In [1]:
import pandas as pd
import re
import os

df = pd.read_csv('ajebo_finance_loan_applications_CLEANED.csv', parse_dates=['Application_Date'])
df.shape


(356, 18)

---
## 1. Why this matters

Nobody hiring an analyst asks to see your code before they see your portfolio link. A GitHub
repository is often the first thing an employer checks — before the CV is read closely. An
analysis that lives only on a trainee's laptop, however good, does not exist to an employer.
This topic makes this week's work visible, understandable, and credible to someone who has never
met the trainee and has ninety seconds to decide whether to keep reading.

---
## 2. GitHub basics

Three concepts are enough to complete this topic:

- **Repository ("repo").** A project folder that lives on GitHub, with a full history of changes.
  One repo per project is the right default for portfolio work.
- **Commit.** A saved snapshot of changes, with a short message describing what changed. Commit
  messages are part of the portfolio too — "fixed stuff" reads very differently to an employer
  than "clean duplicate applications and standardise branch names (Module 6.1)."
- **README.** A file GitHub automatically displays on the repository's front page. For a
  portfolio project, the README *is* the pitch — assume most visitors read only the README.

**Minimum steps to publish this week's project:**

1. Create a new repository (e.g. `ajebo-loan-analysis`), set to public.
2. Add the cleaned dataset — *after* running the data-safety check in Section 3.
3. Add the Module 6.1-6.3 notebooks.
4. Add a `README.md` at the root (Section 4 builds one for real, below).
5. Commit with a clear message and push.
6. Copy the repository's URL — this is the portfolio link.

---
## 3. The data-safety check — before anything else

This is not optional, and it is not new — it is Module 6.1's data-protection discipline, applied
to a *public* space this time. Before uploading any dataset, scan it programmatically. Below is a
reusable checker: it flags suspicious column names, and scans string columns for email- and
phone-shaped values.

In [2]:
SUSPICIOUS_COLUMN_KEYWORDS = [
    'name', 'email', 'phone', 'mobile', 'address', 'ssn', 'nin', 'bvn',
    'passport', 'dob', 'date_of_birth', 'account_number', 'card_number',
]
EMAIL_PATTERN = re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+')
PHONE_PATTERN = re.compile(r'(\+?\d[\d\-\s]{8,}\d)')

def data_safety_check(dataframe, sample_rows=50):
    """Scan a DataFrame for likely personal data before it's published. Prints a report;
    does not modify the data. Always review the output yourself - this is a first pass,
    not a guarantee."""
    print(f"Scanning {len(dataframe)} rows, {len(dataframe.columns)} columns...\n")
    flags = []

    # 1. Column name check
    for col in dataframe.columns:
        for kw in SUSPICIOUS_COLUMN_KEYWORDS:
            if kw in col.lower():
                flags.append(f"Column '{col}' name contains suspicious keyword '{kw}'")

    # 2. Pattern check - each value checked individually, never on a joined mega-string
    #    (joining values before matching can create false cross-value matches)
    sample = dataframe.head(sample_rows)
    for col in dataframe.select_dtypes(include='object').columns:
        values = sample[col].dropna().astype(str)
        if values.apply(lambda v: bool(EMAIL_PATTERN.search(v))).any():
            flags.append(f"Column '{col}' contains values matching an email pattern")
        if values.apply(lambda v: bool(PHONE_PATTERN.search(v))).any():
            flags.append(f"Column '{col}' contains values matching a phone-number pattern")

    if flags:
        print("FLAGGED - do not publish until reviewed:")
        for f in flags:
            print(f"  - {f}")
    else:
        print("No obvious personal-data patterns detected.")
        print("This is a first-pass check, not a guarantee - always review manually too.")
    return flags

_ = data_safety_check(df)


Scanning 356 rows, 18 columns...

No obvious personal-data patterns detected.
This is a first-pass check, not a guarantee - always review manually too.


**Proving the checker actually works, not just trivially passing:** let's deliberately add a
fake row with an obvious email address and confirm the checker catches it.

In [3]:
test_df = df.head(3).copy()
test_df['Contact_Email'] = ['not.a.real.person@example.com', None, None]

flags = data_safety_check(test_df)
assert len(flags) > 0, "Checker should have flagged the injected email column"
print(f"\nConfirmed: checker correctly flagged {len(flags)} issue(s) in the test data.")


Scanning 3 rows, 19 columns...

FLAGGED - do not publish until reviewed:
  - Column 'Contact_Email' name contains suspicious keyword 'email'
  - Column 'Contact_Email' contains values matching an email pattern

Confirmed: checker correctly flagged 2 issue(s) in the test data.


AJEBO's real dataset passes clean because Module 6.1's dataset design guaranteed no real
personal data (`Application_ID` is a synthetic reference). **Run this exact check on any new
dataset before it goes anywhere public** — a clean pass here is necessary, but always pair it
with your own manual look at the columns and a sample of rows.

---
## 4. Anatomy of a strong analyst README

A README that gets read follows a predictable structure, most-convincing-material-first, because
most visitors skim:

1. **Project title and one-line summary.**
2. **Business problem** — two to three sentences: what question, for whom.
3. **Data source** — where it came from, its size, and a one-line data-safety confirmation.
4. **Method** — a short numbered summary, linking to each notebook.
5. **Key findings** — two to four sharp insights (Module 6.3's work), each with its supporting
   chart **embedded**, not just linked.
6. **Tools used.**
7. **How to reproduce.**
8. **Contact / links.**

Rather than just describing this structure, let's build a real one from the actual verified
numbers in this dataset — the same discipline as Module 6.3's traceable narratives, applied to
a README this time.

---
## 5. Generating a real README from real data

Every number below is computed here, not typed from memory — if the underlying data changes,
re-running this cell regenerates an accurate README automatically.

In [4]:
# Recompute the headline findings this README will report (Module 6.2/6.3 verified figures)
branch_stats = df.groupby('Branch_Region').agg(
    n=('Application_ID', 'count'),
    approval_rate=('Loan_Status', lambda s: (s == 'Approved').mean())
).sort_values('approval_rate')
kano = branch_stats.loc['Kano']
next_lowest = branch_stats.iloc[1]

income_loan_corr = df['Monthly_Income_NGN'].corr(df['Loan_Amount_NGN'])

readme_stats = {
    'total_rows': len(df),
    'n_branches': df['Branch_Region'].nunique(),
    'kano_rate': kano.approval_rate * 100,
    'kano_n': int(kano.n),
    'next_lowest_name': next_lowest.name,
    'next_lowest_rate': next_lowest.approval_rate * 100,
    'gap_points': (next_lowest.approval_rate - kano.approval_rate) * 100,
    'income_loan_corr': income_loan_corr,
}
readme_stats


{'total_rows': 356, 'n_branches': 6, 'kano_rate': np.float64(50.0), 'kano_n': 32, 'next_lowest_name': 'Ibadan', 'next_lowest_rate': np.float64(73.17073170731707), 'gap_points': np.float64(23.17073170731707), 'income_loan_corr': np.float64(0.721213799728812)}

In [5]:
readme_content = f'''# AJEBO Finance Loan Approval Analysis

Analysis of {readme_stats["total_rows"]} home loan applications for a fictional Nigerian
microfinance bank, identifying where lending outcomes differ across branches and why.

## Business problem
AJEBO Finance's credit committee wanted to know whether Kano branch's lending criteria needed
review, and what other patterns in the loan book were worth their attention.

## Data
`ajebo_finance_loan_applications_CLEANED.csv` -- {readme_stats["total_rows"]} applications across
{readme_stats["n_branches"]} branches. Application_ID is a synthetic reference; no real personal
data is present (verified with the data-safety check in this repository's notebooks).

## Method
1. [Data cleaning](./01_cleaning.ipynb) -- deduplication, standardisation, validation
   (17-step documented workflow)
2. [Visualisation](./02_visualisation.ipynb) -- chart-by-chart before/after critique
3. [Insight narrative](./03_narrative.ipynb) -- findings translated into recommendations

## Key findings

**Kano's approval rate ({readme_stats["kano_rate"]:.0f}%) is the lowest of {readme_stats["n_branches"]} branches**,
{readme_stats["gap_points"]:.0f} points below the next-lowest branch ({readme_stats["next_lowest_name"]},
{readme_stats["next_lowest_rate"]:.0f}%) -- but Kano's small sample (n={readme_stats["kano_n"]}) means the
true rate could plausibly be much higher. Recommend a manual file review before any policy change.

![Kano approval rate with confidence interval](./images/kano_approval_rate.png)

**Income and loan amount correlate strongly (r={readme_stats["income_loan_corr"]:.2f})** across both
employment types, meaning income is a reasonable first check on an unusually large loan request.

![Income vs loan amount](./images/income_vs_loan.png)

## Tools
Python, pandas, matplotlib, seaborn, Jupyter.

## Reproduce this analysis
```bash
pip install pandas matplotlib seaborn
jupyter notebook 01_cleaning.ipynb
```

## Contact
[Name] - [email or LinkedIn]
'''

with open('README.md', 'w') as f:
    f.write(readme_content)

print(f"README.md written: {len(readme_content)} characters, {readme_content.count(chr(10))} lines")


README.md written: 1709 characters, 45 lines


**Rendered preview of the generated README** (this is exactly what GitHub would show on the
repository's front page):

In [6]:
with open('README.md') as f:
    readme_text = f.read()

try:
    from IPython.display import Markdown, display
    display(Markdown(readme_text))
except ImportError:
    # Falls back to a plain print outside a Jupyter/IPython environment
    print(readme_text)


# AJEBO Finance Loan Approval Analysis

Analysis of 356 home loan applications for a fictional Nigerian
microfinance bank, identifying where lending outcomes differ across branches and why.

## Business problem
AJEBO Finance's credit committee wanted to know whether Kano branch's lending criteria needed
review, and what other patterns in the loan book were worth their attention.

## Data
`ajebo_finance_loan_applications_CLEANED.csv` -- 356 applications across
6 branches. Application_ID is a synthetic reference; no real personal
data is present (verified with the data-safety check in this repository's notebooks).

## Method
1. [Data cleaning](./01_cleaning.ipynb) -- deduplication, standardisation, validation
   (17-step documented workflow)
2. [Visualisation](./02_visualisation.ipynb) -- chart-by-chart before/after critique
3. [Insight narrative](./03_narrative.ipynb) -- findings translated into recommendations

## Key findings

**Kano's approval rate (50%) is the lowest of 6 branches**

**Look closely at the business-problem paragraph** — it names "Kano" directly, typed by hand,
while the rest of the README (the approval rate, the gap, the sample size) is pulled dynamically
from `readme_stats`. That's a real, common drafting inconsistency: if next quarter's data showed
Port Harcourt as the new lowest-performing branch, re-running this cell would silently update
every number in the Key Findings section but leave the business-problem paragraph still naming
Kano by name. **A README that renders without a Python error is not the same as a README that's
actually correct** — always re-read generated text after the underlying data changes, don't just
trust that "it ran." Corrected below: the branch name is now pulled from `readme_stats` too, so
the whole document updates consistently if the underlying data changes.

In [7]:
lowest_branch_name = branch_stats.index[0]  # drive the business-problem line from data, not a typed name

readme_content = readme_content.replace(
    "whether Kano branch's lending criteria needed",
    f"whether {lowest_branch_name} branch's lending criteria needed"
)
with open('README.md', 'w') as f:
    f.write(readme_content)
print(f"README.md corrected: business problem now names '{lowest_branch_name}' dynamically, matching the Key Findings section.")


README.md corrected: business problem now names 'Kano' dynamically, matching the Key Findings section.


---
## 6. What NOT to upload

- **No real personal data**, ever — checked programmatically in Section 3, and manually too.
- **No API keys, passwords, or credentials** — check notebook outputs and code cells for
  anything pasted in during testing.
- **No employer or client data**, unless explicit written permission to publish has been given.

---
## 7. Before-and-after critique — weak vs strong README

**Weak README (a common first attempt):**

> # Project
> Some analysis of loan data. See notebook.ipynb for code.

*Diagnose it:* no business problem, no findings, no visuals, no indication of what's inside
without opening the notebook. An employer scrolls past this in under five seconds.

**Strong README:** the one generated in Section 5 — a one-line summary, a stated business
problem, embedded chart images, and named, sharp, number-backed findings a reader can absorb
without opening any code at all.

**The rule:** *write the README as if the reader will never open the notebook. If it can't stand
completely on its own, it isn't finished.*

---
## 8. Practical activity

1. Run the data-safety check (Section 3) against your own Module 6.1 cleaned file.
2. Extend the `readme_stats` dictionary in Section 5 with one more finding from Module 6.3
   (e.g. the education approval gap or the self-employed loan-size gap), and regenerate the
   README to include it as a third key finding.
3. Proofread your generated README exactly as demonstrated above — render it, read it, and fix
   any leftover drafting artefacts before considering it final.
4. Create a real GitHub repository, add this README and your Module 6.1-6.3 notebooks, and commit
   with a descriptive message.
5. Swap portfolio links with a partner and review each other's README against the checklist
   below.

In [8]:
# Space for your extended readme_stats and regenerated README




---
## 9. Common mistakes

- **A README that's just a table of contents.** Links with no summary force the reader to do all
  the work the README should have done.
- **Screenshots instead of real embedded images.** Embed the actual exported chart file, not a
  lower-quality screenshot of it.
- **Committing a giant, undocumented raw data dump** with no explanation of what it is.
- **Forgetting the reproduce-it-yourself section.**
- **Trusting a generated README without reading it.** Section 5's deliberate bug is the example —
  code that runs without error can still produce text that reads badly or wrongly.

---
## 10. Portfolio checklist

- [ ] Repository is public and the link works when tested in a private/incognito browser window.
- [ ] The data-safety check (Section 3) was run and passed before any data was uploaded.
- [ ] README states the business problem in the first few lines.
- [ ] At least one chart image is embedded directly, not just linked.
- [ ] At least one sharp insight (not just a finding) is stated in plain language, with its
      number.
- [ ] No real personal data, credentials, or unlicensed third-party data is present anywhere.
- [ ] A stranger could reproduce the analysis using only the README's instructions.
- [ ] The README has actually been read start to finish after generating or writing it.

---
## 11. Knowledge check — 5 questions

1. Why does the README matter more than the code for a portfolio project?
2. Name three things that must never be uploaded to a public portfolio repository.
3. What's the difference between linking to a chart image and embedding it?
4. Why did the data-safety checker in Section 3 initially flag AJEBO's own `Dependents` column,
   and what did that reveal about how the checker was written?
5. Why is rendering and reading a generated README not an optional step?

<details><summary><b>Answer key</b></summary>

1. Most visitors, including employers, read the README and never open the code.
2. Any three of: real personal data, API keys/credentials, employer/client data without
   permission, unlicensed third-party data.
3. A linked image requires a click to view; an embedded image displays directly on the page.
4. Joining many short column values into one string before pattern-matching let the phone-number
   regex match *across* multiple values (e.g. "0 1 2 1 0 3" looked like a phone number once
   joined) — it revealed that automated checks need to be verified against real data themselves,
   not just trusted because they returned an answer.
5. Code executing without error only proves the code ran — it says nothing about whether the
   resulting text is accurate, well-formed, or free of leftover drafting artefacts, as
   demonstrated directly in Section 5.

</details>

---
## Next: Module 6.5 — AI-Assisted Exploration and Drafting

The portfolio is now public and documented. Module 6.5 returns to the same AJEBO analysis one
more time — this time with an AI assistant doing first-draft exploration and narrative writing,
and the trainee doing the verification.